# Chapter 1: Introduction to Preprocessing for Text
**Module 03 - Deep Learning for Text with PyTorch**  
*Source integrated from `chapter1.pdf`*

Text models do not learn directly from raw strings. This chapter builds the complete path from messy text to numeric tensors: preprocessing, classical encodings, and a reusable PyTorch `Dataset`/`DataLoader` pipeline.


## Learning Objectives

By the end of this notebook, you will be able to:

- Explain why raw text must be cleaned, tokenized, normalized, and encoded before modeling.
- Apply tokenization, stop word removal, stemming, and rare-word filtering.
- Compare one-hot encoding, bag-of-words, TF-IDF, and embeddings.
- Build a reusable text processing pipeline that produces PyTorch batches.


## 1.1 Course Roadmap and Prerequisites

This module uses text tasks to connect traditional NLP preprocessing with deep learning models.

| Topic | Why it matters |
|---|---|
| Text classification | Assign labels such as sentiment, spam, or topic. |
| Text generation | Predict new words, characters, or sequences. |
| Encoding | Convert language into numeric representations. |
| Deep learning for text | Use CNNs, RNNs, LSTMs, GRUs, Transformers, and transfer learning. |
| Model protection | Understand robustness and adversarial risks. |

**Prerequisites:** PyTorch modeling basics, training and evaluation loops, CNNs, and RNNs.


## 1.2 Text Processing Pipeline

A practical text pipeline turns raw language into model-ready numeric data:

```text
Raw Text -> Clean Text -> Tokenization -> Stop Word Removal -> Stemming -> Rare Word Removal -> Encoding -> Dataset -> DataLoader -> Model
```

**Why this matters:** preprocessing reduces noise, shrinks the feature space, and creates cleaner, more representative datasets.


## 1.3 PyTorch, TorchText, and NLTK

The PDF introduces two core tools:

| Tool | Role in the pipeline |
|---|---|
| `torch` / PyTorch | Tensor operations, datasets, dataloaders, and models. |
| `torchtext` | Tokenizers and text utilities. |
| `nltk` | Stop words, stemming, and frequency-based preprocessing. |

> **Note:** Some notebook environments may not have `torchtext`, `nltk`, or `sklearn` installed. The code below keeps the PDF logic intact and adds small fallbacks or setup comments where useful.


## 1.4 Tokenization

Tokenization extracts words, subwords, punctuation, or characters from raw text. It is the first concrete step toward converting text into features.


In [ ]:
# PDF snippet: tokenization using torchtext
# Install if needed: pip install torchtext
try:
    from torchtext.data.utils import get_tokenizer
    tokenizer = get_tokenizer("basic_english")
except Exception:
    import re

    def tokenizer(text):
        return re.findall(r"\b\w+\b|[^\w\s]", text.lower())

text = "I am reading a book now. I love to read books!"
tokens = tokenizer(text)

print("Original:", text)
print("Tokens:", tokens)
print("Count:", len(tokens))


## 1.5 Stop Word Removal

Stop words are very frequent words such as `a`, `the`, `and`, or `or`. Removing them can:

- Reduce vocabulary size.
- Keep attention on more informative terms.
- Reduce the number of low-value features.

Stop word removal is helpful for many classical pipelines, but it is not always used with modern contextual models because words such as `not` can be important.


In [ ]:
# PDF snippet: stop word removal with NLTK
# Install if needed: pip install nltk
import nltk
from nltk.corpus import stopwords

try:
    stop_words = set(stopwords.words("english"))
except LookupError:
    nltk.download("stopwords", quiet=True)
    stop_words = set(stopwords.words("english"))

tokens = ["I", "am", "reading", "a", "book", "now", ".", "I", "love", "to", "read", "books", "!"]
filtered_tokens = [token for token in tokens if token.lower() not in stop_words]

print("Before:", tokens)
print("After: ", filtered_tokens)


## 1.6 Stemming

Stemming reduces inflected words to a base-like form.

| Original forms | Stemmed form |
|---|---|
| `running`, `runs`, `ran` | `run` |
| `reading`, `read` | `read` |
| `books` | `book` |

Stemming is rule-based and sometimes produces non-dictionary stems, but it is fast and useful for vocabulary reduction.


In [ ]:
# PDF snippet: stemming with PorterStemmer
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()
filtered_tokens = ["reading", "book", ".", "love", "read", "books", "!"]
stemmed_tokens = [stemmer.stem(token) for token in filtered_tokens]

print("Before stemming:", filtered_tokens)
print("After stemming: ", stemmed_tokens)


## 1.7 Rare Word Removal

Rare words can add sparsity without adding much generalizable signal. The PDF uses `FreqDist` with a threshold to keep only frequent tokens.

> **Tip:** The threshold should be tuned to the size of the corpus. A high threshold on a tiny example may remove everything.


In [ ]:
# PDF snippet: rare word removal with NLTK FreqDist
from nltk.probability import FreqDist

stemmed_tokens = ["read", "book", ".", "love", "read", "book", "!"]
freq_dist = FreqDist(stemmed_tokens)
threshold = 1
common_tokens = [token for token in stemmed_tokens if freq_dist[token] > threshold]

print("Frequencies:", dict(freq_dist))
print("Common tokens:", common_tokens)


## 1.8 Encoding Text Data

Text encoding converts tokens into machine-readable numbers so models can analyze patterns.

| Technique | Captures | Loses | Typical use |
|---|---|---|---|
| One-hot encoding | Token identity | Similarity and order | Small vocab demos, simple baselines |
| Bag-of-words | Word frequency | Word order | Classical ML baselines |
| TF-IDF | Frequency weighted by rarity | Word order and semantics | Search and sparse text classification |
| Embeddings | Dense semantic representation | Depends on training quality | Neural networks and modern NLP |

The PDF recommends choosing one encoding technique for a given pipeline to avoid redundant features.


### One-Hot Encoding

Each vocabulary item receives a unique binary vector. For `["cat", "dog", "rabbit"]`, the vectors are:

| Word | Vector |
|---|---|
| `cat` | `[1, 0, 0]` |
| `dog` | `[0, 1, 0]` |
| `rabbit` | `[0, 0, 1]` |


In [ ]:
# PDF snippet: one-hot encoding with PyTorch
import torch

vocab = ["cat", "dog", "rabbit"]
vocab_size = len(vocab)
one_hot_vectors = torch.eye(vocab_size)
one_hot_dict = {word: one_hot_vectors[i] for i, word in enumerate(vocab)}

for word, vector in one_hot_dict.items():
    print(f"{word:>6} -> {vector.tolist()}")


### Bag-of-Words

Bag-of-words treats a document as an unordered collection of words. For `"The cat sat on the mat"`, a normalized count view is:

```python
{"the": 2, "cat": 1, "sat": 1, "on": 1, "mat": 1}
```


In [ ]:
# PDF snippet: CountVectorizer for bag-of-words
# Install if needed: pip install scikit-learn
from sklearn.feature_extraction.text import CountVectorizer

corpus = [
    "This is the first document.",
    "This document is the second document.",
    "And this is the third one.",
    "Is this the first document?",
]

count_vectorizer = CountVectorizer()
bow_matrix = count_vectorizer.fit_transform(corpus)

print("Vocabulary:", count_vectorizer.get_feature_names_out())
print("BoW matrix:")
print(bow_matrix.toarray())


### TF-IDF

TF-IDF means **Term Frequency - Inverse Document Frequency**. It scores words by balancing:

- **Term frequency:** how often a word appears in a document.
- **Inverse document frequency:** how rare the word is across documents.

Rare but informative words receive higher scores; common words receive lower scores.


In [ ]:
# PDF snippet: TfidfVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(corpus)

print("Vocabulary:", tfidf_vectorizer.get_feature_names_out())
print("TF-IDF matrix:")
print(tfidf_matrix.toarray().round(3))


## 1.9 Vocabulary Building

For neural models, a common bridge between preprocessing and embeddings is a word-to-index vocabulary.


In [ ]:
from collections import Counter

corpus = [
    "I love deep learning",
    "PyTorch is a great deep learning framework",
    "Natural language processing with deep learning",
]

all_tokens = []
for doc in corpus:
    all_tokens.extend(tokenizer(doc))

freq = Counter(all_tokens)
vocab = {"<PAD>": 0, "<UNK>": 1}
for word, count in freq.most_common():
    if count >= 1:
        vocab[word] = len(vocab)

print("Word frequencies:", freq.most_common(10))
print("Vocabulary size:", len(vocab))
print("Vocabulary:", vocab)


## 1.10 Dataset and DataLoader Recap

The PDF closes by putting preprocessing and encoding into a pipeline. PyTorch uses:

| Component | Responsibility |
|---|---|
| `Dataset` | Stores processed examples and returns one item at a time. |
| `DataLoader` | Batches, shuffles, and can parallelize data loading. |


In [ ]:
# PDF snippet: implementing Dataset and DataLoader
from torch.utils.data import Dataset, DataLoader


class TextDataset(Dataset):
    def __init__(self, text):
        self.text = text

    def __len__(self):
        return len(self.text)

    def __getitem__(self, idx):
        return self.text[idx]


## 1.11 Helper Functions for a Text Processing Pipeline

The PDF uses helper functions to:

1. Extract sentences from raw text.
2. Preprocess each sentence.
3. Encode processed sentences with `CountVectorizer`.
4. Wrap encoded rows in a `Dataset` and `DataLoader`.


In [ ]:
# PDF snippets: helper functions and complete pipeline
import re
from sklearn.feature_extraction.text import CountVectorizer


def extract_sentences(data):
    return re.findall(r"[A-Z][^.!?]*[.!?]", data)


def preprocess_sentences(sentences, min_count=1):
    processed_sentences = []

    for sentence in sentences:
        sentence = sentence.lower()
        sentence_tokens = tokenizer(sentence)
        sentence_tokens = [
            token for token in sentence_tokens
            if token.isalpha() and token not in stop_words
        ]
        sentence_tokens = [stemmer.stem(token) for token in sentence_tokens]

        freq_dist = FreqDist(sentence_tokens)
        sentence_tokens = [
            token for token in sentence_tokens
            if freq_dist[token] >= min_count
        ]
        processed_sentences.append(" ".join(sentence_tokens))

    return processed_sentences


def encode_sentences(sentences):
    vectorizer = CountVectorizer()
    encoded_sentences = vectorizer.fit_transform(sentences).toarray()
    return encoded_sentences, vectorizer


def text_processing_pipeline(text, batch_size=2):
    sentences = extract_sentences(text)
    processed = preprocess_sentences(sentences)
    encoded_sentences, vectorizer = encode_sentences(processed)
    dataset = TextDataset(torch.tensor(encoded_sentences, dtype=torch.float32))
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    return dataloader, vectorizer, processed


In [ ]:
# PDF snippet: applying the text processing pipeline
text_data = "This is the first text data. And here is another one."
dataloader, vectorizer, processed = text_processing_pipeline(text_data)

print("Processed sentences:", processed)
print("Vocabulary:", vectorizer.get_feature_names_out())
print("First batch:")
print(next(iter(dataloader)))


## Chapter Summary

| Stage | Core idea | Representative tool |
|---|---|---|
| Tokenization | Split raw text into tokens | `torchtext` or regex fallback |
| Stop word removal | Remove very common words | `nltk.corpus.stopwords` |
| Stemming | Reduce words to root-like forms | `PorterStemmer` |
| Rare word removal | Reduce sparse, low-frequency terms | `FreqDist` |
| One-hot | Unique vector per token | `torch.eye` |
| Bag-of-words | Count word occurrences | `CountVectorizer` |
| TF-IDF | Weight words by informativeness | `TfidfVectorizer` |
| Pipeline | Batch encoded examples for modeling | `Dataset`, `DataLoader` |

You now have the preprocessing and encoding foundation needed for text classification and generation models.
